In [ ]:
# ==== Core Libraries ====
import os
import copy
import random
from collections import Counter
from datetime import datetime

# ==== Data Processing ====
import numpy as np
import pandas as pd
from scipy import stats

# ==== PyTorch ====
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.amp import autocast, GradScaler

# ==== Visualization ====
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, input_channel, features_d, num_classes, num_aps, input_width):
        super(Discriminator, self).__init__()
        self.num_classes = num_classes
        self.num_aps = num_aps
        self.input_width = input_width
        self.input_channel = input_channel

        if self.num_aps == 11:
            last_kernal_size = 12
            last_stride = 5
            last_padding = 4

        self.feature_extractor = nn.Sequential(
            nn.Conv2d(self.input_channel, features_d, kernel_size=4, stride=1, padding=4),
            nn.LeakyReLU(0.2),
            self._block(features_d, features_d * 2, 10, 1, 4),
            self._block(features_d * 2, features_d * 4, 10, 1, 4),
            self._block(features_d * 4, features_d * 8, 10, 2, 4),
        )

        self.classifier = nn.Conv2d(features_d * 8, num_classes + 1,
                                    kernel_size=last_kernal_size,
                                    stride=last_stride,
                                    padding=last_padding)

    def _block(self, in_channels, out_channels, kernel_size, stride, padding):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False),
            nn.InstanceNorm2d(out_channels, affine=True),
            nn.LeakyReLU(0.2),
        )

    def forward(self, x):
        features = self.feature_extractor(x)  # shape: [B, F, H, W]
        logits = self.classifier(features)    # shape: [B, K+1, 1, 1]
        return logits.view(x.size(0), -1)     # shape: [B, K+1]


class Generator(nn.Module):
  def __init__(self,
                channels_noise,
                input_channel,
                features_g,
                num_classes,
                num_aps,
                input_width,
                embed_size):
    super(Generator, self).__init__()
    self.num_aps = num_aps
    self.input_channel = input_channel
    self.input_width = input_width
    self.embed_size = embed_size
    self.net = nn.Sequential(
      self._block(channels_noise + embed_size, features_g * 8, 10, 1, 4),  # img: 12x21
      self._block(features_g * 8, features_g * 4, 10, 1, 4),  # img: 13x22
      self._block(features_g * 4, features_g * 2, 10, 1, 4),  # img: 14x23
      self._block(features_g * 2, features_g, 10, 1, 4),  # img: 15x24
      nn.ConvTranspose2d(features_g, input_channel, kernel_size=9, stride=1, padding=6),
      nn.Tanh(),
      #nn.ReLU()
    )
    self.embed = nn.Embedding(num_classes, embed_size * num_aps * input_width)


  def _block(self, in_channels, out_channels, kernel_size, stride, padding):
    return nn.Sequential(
      nn.ConvTranspose2d(
          in_channels, out_channels, kernel_size, stride, padding, bias=False,
      ),
      nn.BatchNorm2d(out_channels),
      nn.ReLU(),
    )

  def forward(self, x, labels):
    # latent vector z: N x noise_dim x 1 x 1
    embedding = self.embed(labels).view(labels.shape[0], self.embed_size, self.num_aps, self.input_width)
    x = torch.cat([x, embedding], dim=1)  # N x C x img_size(H) x img_size(W)
    return self.net(x)


def initialize_weights(model):
  # Initializes weights according to the DCGAN paper
  for m in model.modules():
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.BatchNorm2d)):
      nn.init.normal_(m.weight.data, 0.0, 0.02)

In [ ]:
def MinMaxScaler(data):
  mask = data != -120.0
  scaled = data.copy()

  min_val = np.min(scaled[mask])
  max_val = np.max(scaled[mask])

  scaled[mask] = 2 * (scaled[mask] - min_val) / (max_val - min_val + 1e-7) - 1

  scaled[~mask] = -1.0

  return scaled, min_val, max_val

def windowing(ori_data, y, seq_len = 20, hop_size = 10, shuffle=True):
    windowed_data = []
    windowed_label = []
    # Cut data by sequence length
    for i in range(0, len(ori_data) - seq_len, hop_size):
        _x = ori_data[i:i + seq_len]
        _y = stats.mode(y[i:i + seq_len], axis=None, keepdims=True).mode[0]
        windowed_data.append(_x)
        windowed_label.append(_y)
    if shuffle:
        idx = np.random.permutation(len(windowed_data))
        data = []
        label = []
        for i in range(len(windowed_data)):
            data.append(windowed_data[idx[i]])
            label.append(windowed_label[idx[i]])
    else:
        data = windowed_data
        label = windowed_label
    data = np.asarray(data)
    label = np.asarray(label)

    return data, label

def get_gradient_norm(model):
  total_norm = 0.0
  for p in model.parameters():
    if p.grad is not None:
      param_norm = p.grad.data.norm(2)
      total_norm += param_norm.item() ** 2
  return total_norm ** 0.5

In [ ]:
INPUT_CSV = "/content/drive/MyDrive/rssi_project/ssl_gan/training_dataset.csv"
#house_map = {0:0, 1:1, 2:2}
house_map = {"ssl_training":0}
data = np.loadtxt(INPUT_CSV, delimiter=",", skiprows=1)
ori_data = data[:, :-1]
label = data[:, -1]
norm_data, min_val, max_val = MinMaxScaler(ori_data)

In [ ]:
rssi_fp, y_fp = windowing(norm_data, label, seq_len=20, hop_size=10)
y_fp = y_fp-1
rssi_fp = np.transpose(rssi_fp, (0, 2, 1))

In [ ]:
np.unique(y_fp)

array([0., 1., 2.])

In [ ]:
X_train = rssi_fp
y_train = y_fp
X_train2D = torch.from_numpy(X_train)
X_train2D = X_train2D.unsqueeze(1)

X_train = torch.from_numpy(X_train)
y_train = torch.from_numpy(y_train).int()

In [ ]:
torch.manual_seed(0)
torch.cuda.manual_seed(0)
np.random.seed(0)
random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
LEARNING_RATE = 1e-4
BATCH_SIZE = 64
WINDOW_SIZE = 20
CHANNELS_RSSI = 1
GEN_EMBEDDING = 100
Z_DIM = 11
FEATURES_DISC = 32
FEATURES_GEN = 32
DISC_ITERATIONS = 3
NUM_CLASSES = 3
APs = 11

In [ ]:
dataset_train = torch.utils.data.TensorDataset(X_train2D.float(), y_train)
loader_train = torch.utils.data.DataLoader(dataset_train,
                                         batch_size=BATCH_SIZE,
                                         shuffle=True,
                                         drop_last=True,)

In [ ]:
gen = Generator(Z_DIM, CHANNELS_RSSI, FEATURES_GEN, NUM_CLASSES, APs, WINDOW_SIZE, GEN_EMBEDDING).to(device)
disc = Discriminator(CHANNELS_RSSI, FEATURES_DISC, NUM_CLASSES, APs, WINDOW_SIZE).to(device)

initialize_weights(gen)
initialize_weights(disc)

In [ ]:
# initializate optimizer
opt_gen = optim.Adam(gen.parameters(), lr=LEARNING_RATE, betas=(0.0, 0.9))
opt_disc = optim.Adam(disc.parameters(), lr=LEARNING_RATE, betas=(0.0, 0.9))

scheduler_gen = StepLR(opt_gen, step_size=100, gamma=0.5)
scheduler_disc = StepLR(opt_disc, step_size=100, gamma=0.5)

In [ ]:
scaler = GradScaler()

In [ ]:
# for tensorboard plotting
fixed_noise = torch.randn(32, Z_DIM, APs, 20).uniform_(0, 1).to(device)
fixed_noise_plot = fixed_noise[0][0]
house_name = "ssl_training"
model_name = "ConGAN_wgp_house_" + house_name
writer_loss = SummaryWriter(f"/content/drive/MyDrive/rssi_project/ssl_gan/{model_name}/loss/")

In [ ]:
checkpoint_path = "/content/drive/MyDrive/rssi_project/ssl_gan/ConGAN_wgp_house_ssl_training_final.pt"
checkpoint = torch.load(checkpoint_path, map_location=device)

gen.load_state_dict(checkpoint['gen_state_dict'])
disc.load_state_dict(checkpoint['disc_state_dict'])

opt_gen.load_state_dict(checkpoint['opt_gen_state_dict'])
opt_disc.load_state_dict(checkpoint['opt_disc_state_dict'])

scaler.load_state_dict(checkpoint['scaler_state_dict'])
scheduler_gen.load_state_dict(checkpoint['scheduler_gen_state_dict'])
scheduler_disc.load_state_dict(checkpoint['scheduler_disc_state_dict'])

start_epoch = checkpoint['epoch'] + 1

In [ ]:
NUM_EPOCHS = 500

gen_loss = []
disc_loss = []
class_acc = []

best_acc = 0

criterion = nn.CrossEntropyLoss()

# Prepare validation set from loader_train
val_data = []
for real, labels in loader_train:
    val_data.append((real.to(device), labels.to(device)))
    if len(val_data) >= 5:  # small subset for monitoring
        break

for epoch in range(start_epoch,start_epoch+200):
    start = time.time()

    gen_total_loss = 0
    disc_total_loss = 0
    num_batches = 0

    for batch_idx, (real, labels) in enumerate(loader_train):
        real = real.to(device)
        labels = labels.to(device).long()
        cur_batch_size = real.size(0)

        # --- Train Discriminator ---
        for _ in range(DISC_ITERATIONS):
            noise = torch.randn(cur_batch_size, Z_DIM, APs, WINDOW_SIZE).uniform_(0, 1).to(device)
            with autocast("cuda"):
                fake = gen(noise, labels)
                real_logits = disc(real)
                fake_logits = disc(fake.detach())
                label_loss = criterion(real_logits, labels)
                fake_loss = criterion(fake_logits, torch.full_like(labels, NUM_CLASSES))
                loss_disc = label_loss + fake_loss

            disc.zero_grad()
            scaler.scale(loss_disc).backward(retain_graph=True)
            scaler.unscale_(opt_disc)
            torch.nn.utils.clip_grad_norm_(disc.parameters(), max_norm=10.0)
            grad_norm_disc = get_gradient_norm(disc)
            writer_loss.add_scalar("GradNorm/disc", grad_norm_disc, epoch * len(loader_train) + batch_idx)
            scaler.step(opt_disc)
            scaler.update()

        # --- Train Generator ---
        noise = torch.randn(cur_batch_size, Z_DIM, APs, WINDOW_SIZE).uniform_(0, 1).to(device)
        with autocast("cuda"):
            fake = gen(noise, labels)
            fake_logits = disc(fake)
            loss_gen = criterion(fake_logits, labels)

        gen.zero_grad()
        scaler.scale(loss_gen).backward()
        scaler.unscale_(opt_gen)
        torch.nn.utils.clip_grad_norm_(gen.parameters(), max_norm=10.0)
        grad_norm_gen = get_gradient_norm(gen)
        writer_loss.add_scalar("GradNorm/generator_batch", grad_norm_gen, epoch * len(loader_train) + batch_idx)
        scaler.step(opt_gen)
        scaler.update()

        gen_total_loss += loss_gen.item()
        disc_total_loss += loss_disc.item()
        num_batches += 1

    avg_loss_gen = gen_total_loss / num_batches
    avg_loss_disc = disc_total_loss / num_batches
    gen_loss.append(avg_loss_gen)
    disc_loss.append(avg_loss_disc)
    scheduler_disc.step()
    scheduler_gen.step()

    # --- Evaluation ---
    correct = 0
    total = 0
    with torch.no_grad():
        for x_val, y_val in val_data:
            logits = disc(x_val)
            pred = logits.argmax(dim=1)
            correct += (pred == y_val).sum().item()
            total += y_val.size(0)
    acc = correct / total
    class_acc.append(acc)
    writer_loss.add_scalar("Eval/class_acc", acc, epoch)

    curr_time = time.time()
    elapsed_time = int(curr_time - start)
    formatted_time = datetime.now().strftime("%H:%M:%S")

    print(
        f"Time: {formatted_time}, Time Elapsed: {elapsed_time} sec, Epoch [{epoch}/{NUM_EPOCHS}] "
        f"Loss D: {avg_loss_disc:.4f}, Loss G: {avg_loss_gen:.4f}, Acc: {acc:.4f} "
        f"GradNorm Disc: {grad_norm_disc:.4f}, GradNorm Generator: {grad_norm_gen:.4f}"
    )

    if epoch % 20 == 0:
        torch.save({
            'epoch': epoch,
            'gen_state_dict': gen.state_dict(),
            'disc_state_dict': disc.state_dict(),
            'opt_gen_state_dict': opt_gen.state_dict(),
            'opt_disc_state_dict': opt_disc.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'scheduler_gen_state_dict': scheduler_gen.state_dict(),
            'scheduler_disc_state_dict': scheduler_disc.state_dict(),
        }, f"/content/drive/MyDrive/rssi_project/{model_name}_epoch{epoch}.pt")

        with torch.no_grad():
            x_fake = fake[0].view(11, 20)
            x_real = real[0].view(11, 20)
            cur_labels_numpy = labels.cpu().detach().numpy()

            plot_line_rssi_gan(x_fake.cpu().detach().numpy(), cur_labels_numpy[0] + 1, house=house_name,
                house_map=house_map[house_name], ymin=-0.9, ymax=1.1, save=True,
                save_dir='/content/drive/MyDrive/rssi_project/ssl_gan/', model_name=model_name+'-'+str(epoch), transpose=True)
            plot_line_rssi_gan(x_real.cpu().detach().numpy(), cur_labels_numpy[0] + 1, house=house_name,
                house_map=house_map[house_name], ymin=-0.9, ymax=1.1, save=True,
                save_dir='/content/drive/MyDrive/rssi_project/ssl_gan/', model_name=model_name+'-'+str(epoch), transpose=True)

    if epoch == NUM_EPOCHS - 1:
        torch.save({
            'epoch': epoch,
            'gen_state_dict': gen.state_dict(),
            'disc_state_dict': disc.state_dict(),
            'opt_gen_state_dict': opt_gen.state_dict(),
            'opt_disc_state_dict': opt_disc.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'scheduler_gen_state_dict': scheduler_gen.state_dict(),
            'scheduler_disc_state_dict': scheduler_disc.state_dict(),
        }, f"/content/drive/MyDrive/rssi_project/ssl_gan/{model_name}_final_{start_epoch}.pt")


Time: 15:07:00, Time Elapsed: 48 sec, Epoch [500/500] Loss D: 0.2420, Loss G: 3.5766, Acc: 0.8594 GradNorm Disc: 10.0000, GradNorm Generator: 10.0000
Time: 15:07:49, Time Elapsed: 48 sec, Epoch [501/500] Loss D: 0.2403, Loss G: 3.6493, Acc: 0.8406 GradNorm Disc: 10.0000, GradNorm Generator: 10.0000
Time: 15:08:37, Time Elapsed: 48 sec, Epoch [502/500] Loss D: 0.2398, Loss G: 3.6840, Acc: 0.8750 GradNorm Disc: 10.0000, GradNorm Generator: 10.0000
Time: 15:09:26, Time Elapsed: 48 sec, Epoch [503/500] Loss D: 0.2422, Loss G: 3.6809, Acc: 0.8531 GradNorm Disc: 10.0000, GradNorm Generator: 10.0000
Time: 15:10:14, Time Elapsed: 48 sec, Epoch [504/500] Loss D: 0.2421, Loss G: 3.6164, Acc: 0.8719 GradNorm Disc: 10.0000, GradNorm Generator: 10.0000
Time: 15:11:02, Time Elapsed: 48 sec, Epoch [505/500] Loss D: 0.2400, Loss G: 3.5816, Acc: 0.8438 GradNorm Disc: 10.0000, GradNorm Generator: 10.0000
Time: 15:11:50, Time Elapsed: 48 sec, Epoch [506/500] Loss D: 0.2416, Loss G: 3.6114, Acc: 0.8562 Gr

KeyboardInterrupt: 